In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# 1. SYNTHETIC EMAIL DATA
Each row is one email. Features are simple, measurable
properties that distinguish spam from legitimate mail.

Features:
  - num_links:         count of hyperlinks in the email
  - caps_ratio:        fraction of characters that are uppercase
  - num_recipients:    how many people received the email
  - suspicious_words:  count of words like "free", "winner", "click"

In [3]:
np.random.seed(42)
n = 600

# Legitimate emails: few links, normal caps, few recipients, few spam words
legit_links      = np.random.poisson(lam=1, size=n // 2).astype(float)
legit_caps       = np.random.normal(loc=0.05, scale=0.02, size=n // 2)
legit_recipients = np.random.poisson(lam=2, size=n // 2).astype(float)
legit_spam_words = np.random.poisson(lam=1, size=n // 2).astype(float)

# Spam emails: more links, lots of caps, many recipients, many spam words
spam_links      = np.random.poisson(lam=6, size=n // 2).astype(float)
spam_caps       = np.random.normal(loc=0.25, scale=0.08, size=n // 2)
spam_recipients = np.random.poisson(lam=50, size=n // 2).astype(float)
spam_spam_words = np.random.poisson(lam=7, size=n // 2).astype(float)

X = np.column_stack([
    np.concatenate([legit_links, spam_links]),
    np.concatenate([legit_caps, spam_caps]),
    np.concatenate([legit_recipients, spam_recipients]),
    np.concatenate([legit_spam_words, spam_spam_words])
]).astype(np.float32)

y = np.concatenate([
    np.zeros(n // 2),
    np.ones(n // 2)
]).astype(np.float32).reshape(-1, 1)

# Shuffle
shuffle = np.random.permutation(n)
X, y = X[shuffle], y[shuffle]

# Normalize
mean = X.mean(axis=0)
std  = X.std(axis=0)
X = (X - mean) / std

# Train/test split
split = int(0.8 * n)
X_train = torch.tensor(X[:split])
X_test  = torch.tensor(X[split:])
y_train = torch.tensor(y[:split])
y_test  = torch.tensor(y[split:])

print(f"Training samples: {len(y_train)}")
print(f"Test samples:     {len(y_test)}\n")

Training samples: 480
Test samples:     120



# 2. DEFINE THE NETWORK

Same structure as the AND gate code: linear → relu → linear → sigmoid.
The only difference is the input size (4 instead of 2) and the
hidden layer size (8 instead of 4) to handle the extra features.

In PyTorch, the forward pass is explicit. You can see exactly
what happens to the data at each step. TensorFlow's Sequential
model hides this behind a single model.fit() call.

In [19]:
class SpamClassifier(nn.Module):
    def __init__(self):
        super(SpamClassifier, self).__init__()
        self.fc1     = nn.Linear(4, 8)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.relu(self.fc1(x))
        out = self.sigmoid(self.fc2(out))
        return out

model = SpamClassifier()
criterion = nn.BCELoss()
optimizer = optim.RMSprop(model.parameters(), lr=0.01)

# 3. PREDICTIONS BEFORE TRAINING


In [20]:
print("Predictions BEFORE training:")
print(f"  {'Links':>6} {'Caps%':>6} {'Recips':>7} {'SpamW':>6} {'Output':>8} {'Decision':>10}")
print(f"  {'─'*6} {'─'*6} {'─'*7} {'─'*6} {'─'*8} {'─'*10}")

sample_indices = [0, 1, 2, 3, 4, 5]
with torch.no_grad():
    pre_pred = model(X_test[:6]).numpy().flatten()

for i, idx in enumerate(sample_indices):
    real = X_test[idx].numpy() * std + mean
    pred = pre_pred[i]
    label = "Spam" if pred >= 0.5 else "Legit"
    actual = "Spam" if y_test[idx] == 1 else "Legit"
    print(f"  {real[0]:>6.0f} {real[1]:>6.1%} {real[2]:>7.0f} {real[3]:>6.0f} {pred:>7.4f}  {label:>10} (actual: {actual})")

Predictions BEFORE training:
   Links  Caps%  Recips  SpamW   Output   Decision
  ────── ────── ─────── ────── ──────── ──────────
       5  28.2%      46      5  0.4176       Legit (actual: Spam)
       5  19.7%      50     10  0.4185       Legit (actual: Spam)
       1   4.8%       2      1  0.4161       Legit (actual: Legit)
       3  21.0%      35      9  0.4050       Legit (actual: Spam)
       7  43.4%      56     10  0.3968       Legit (actual: Spam)
       9  33.4%      52      6  0.3953       Legit (actual: Spam)


# 4. TRAIN

The training loop in PyTorch is explicit: forward pass, compute
loss, backward pass, update weights. TensorFlow hides this
inside model.fit(). The explicit version is more verbose but
makes it clear what happens at each step.


In [21]:
print("\nTraining...")
train_losses = []

for epoch in range(100):
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    train_losses.append(loss.item())

    if (epoch + 1) % 20 == 0:
        preds = (outputs >= 0.5).float()
        acc = (preds == y_train).float().mean().item()
        print(f"  Epoch {epoch+1:>3}/100 | Loss: {loss.item():.4f} | Train Acc: {acc:.1%}")


Training...
  Epoch  20/100 | Loss: 0.0133 | Train Acc: 100.0%
  Epoch  40/100 | Loss: 0.0048 | Train Acc: 100.0%
  Epoch  60/100 | Loss: 0.0027 | Train Acc: 100.0%
  Epoch  80/100 | Loss: 0.0017 | Train Acc: 100.0%
  Epoch 100/100 | Loss: 0.0012 | Train Acc: 100.0%


# 5. EVALUATE ON TEST SET


In [22]:
with torch.no_grad():
    y_pred_prob = model(X_test).numpy().flatten()
    y_pred = (y_pred_prob >= 0.5).astype(int)
    y_true = y_test.numpy().flatten().astype(int)

tp = np.sum((y_pred == 1) & (y_true == 1))
tn = np.sum((y_pred == 0) & (y_true == 0))
fp = np.sum((y_pred == 1) & (y_true == 0))
fn = np.sum((y_pred == 0) & (y_true == 1))

accuracy  = (tp + tn) / len(y_true)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f"\n  Test Accuracy: {accuracy:.1%}")
print(f"  Precision:     {precision:.1%}  (of predicted spam, how much was actually spam)")
print(f"  Recall:        {recall:.1%}  (of actual spam, how much did we catch)")

print(f"\n  Confusion Matrix:")
print(f"                   Predicted")
print(f"                   Legit   Spam")
print(f"  Actual Legit   [  {tn:>4}     {fp:>4}  ]")
print(f"  Actual Spam    [  {fn:>4}     {tp:>4}  ]")



  Test Accuracy: 100.0%
  Precision:     100.0%  (of predicted spam, how much was actually spam)
  Recall:        100.0%  (of actual spam, how much did we catch)

  Confusion Matrix:
                   Predicted
                   Legit   Spam
  Actual Legit   [    60        0  ]
  Actual Spam    [     0       60  ]


# 6. CLASSIFY NEW EMAILS


In [23]:
new_emails = np.array([
    [0,  0.03,  1,  0],    # personal email, no links
    [1,  0.06,  2,  1],    # normal newsletter
    [8,  0.30, 75,  9],    # obvious spam blast
    [3,  0.15, 20,  4],    # suspicious marketing
    [1,  0.04,  1,  0],    # colleague email
    [12, 0.40, 100, 11],   # extreme spam
], dtype=np.float32)

new_normalized = (new_emails - mean) / std
new_tensor = torch.tensor(new_normalized)

with torch.no_grad():
    new_predictions = model(new_tensor).numpy().flatten()

print(f"\n  New Emails:")
print(f"  {'Links':>6} {'Caps%':>6} {'Recips':>7} {'SpamW':>6} {'Confidence':>12} {'Decision':>10}")
print(f"  {'─'*6} {'─'*6} {'─'*7} {'─'*6} {'─'*12} {'─'*10}")

for email, pred in zip(new_emails, new_predictions):
    label = "Spam" if pred >= 0.5 else "Legit"
    confidence = pred if pred >= 0.5 else 1 - pred
    print(f"  {email[0]:>6.0f} {email[1]:>6.0%} {email[2]:>7.0f} {email[3]:>6.0f} {confidence:>11.1%} {label:>10}")


  New Emails:
   Links  Caps%  Recips  SpamW   Confidence   Decision
  ────── ────── ─────── ────── ──────────── ──────────
       0     3%       1      0      100.0%      Legit
       1     6%       2      1       99.9%      Legit
       8    30%      75      9      100.0%       Spam
       3    15%      20      4       77.8%      Legit
       1     4%       1      0       99.9%      Legit
      12    40%     100     11      100.0%       Spam
